In [1]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import pdfplumber

In [4]:
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [5]:


def extract_pdf_content(pdf_path):
    final_text = []

    # Load PDF
    doc = fitz.open(pdf_path)

    for page_num in range(len(doc)):
        page = doc[page_num]

        # ---- 1. Extract normal text ----
        text = page.get_text()
        if text.strip():
            final_text.append(text)

        # ---- 2. Extract images and OCR ----
        image_list = page.get_images(full=True)

        for img_index, img in enumerate(image_list):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]

            image = Image.open(io.BytesIO(image_bytes))
            ocr_text = pytesseract.image_to_string(image)

            if ocr_text.strip():
                final_text.append(ocr_text)

        # ---- 3. Extract tables ----
        with pdfplumber.open(pdf_path) as pdf:
            table_page = pdf.pages[page_num]
            tables = table_page.extract_tables()

            for table in tables:
                table_text = "\n".join(
                    ["\t".join([str(cell) for cell in row]) for row in table]
                )
                final_text.append(table_text)

    return "\n".join(final_text)

In [6]:
# Usage
file_path = (
    "../datasets/pdf_files/"
    "2023_Jan_7_Feature_Engineering_Techniques.pdf"
)
result = extract_pdf_content(file_path)

In [7]:
with open("output.txt", "w", encoding="utf-8") as f:
    f.write(result)

print("Extraction completed.")

Extraction completed.
